# Ultrasound Domain Adaptation using CycleGAN

Unpaired image-to-image translation (Domain A → Domain B) using a CycleGAN-inspired generator architecture.

**Pipeline:** Data loading → Generator/Discriminator architecture → Training → Inference → Quantitative Evaluation (SSIM, HC, BD, CNR, GSM)

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Input, Model

print("1. Loading Dataset...")
# 1. LOAD THE DATA
all_paths = []
for root, dirs, files in os.walk("/kaggle/input"):
    if any(forbidden in root.lower() for forbidden in ["mask", "annotation", "groundtruth", "segmentation"]):
        continue
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
            if "mask" not in file.lower() and "expert" not in file.lower():
                all_paths.append(os.path.join(root, file))

random.seed(42)
random.shuffle(all_paths)
split_idx = len(all_paths) // 2
paths_a = all_paths[:split_idx]
paths_b = all_paths[split_idx:]

def load_and_preprocess(path):
    path = tf.cast(path, tf.string) 
    image = tf.io.decode_image(tf.io.read_file(path), channels=1, expand_animations=False)
    image = tf.image.resize(image, [400, 400])
    return tf.cast(image, tf.float32) / 255.0

ds_a = tf.data.Dataset.from_tensor_slices(tf.constant(paths_a, dtype=tf.string)).map(load_and_preprocess)
ds_b = tf.data.Dataset.from_tensor_slices(tf.constant(paths_b, dtype=tf.string)).map(load_and_preprocess)
train_dataset = tf.data.Dataset.zip((ds_a, ds_b)).shuffle(100).batch(1)

print("2. Defining AI Architecture...")
# 2. DEFINE THE GENERATOR ARCHITECTURE
class ReflectionPadding2D(layers.Layer):
    def __init__(self, padding=(1, 1), **kwargs):
        super().__init__(**kwargs)
        self.padding = tuple(padding)
    def call(self, x):
        return tf.pad(x, [[0, 0], [self.padding[0], self.padding[0]], [self.padding[1], self.padding[1]], [0, 0]], mode='REFLECT')

def residual_block(x, filters):
    y = ReflectionPadding2D()(x)
    y = layers.Conv2D(filters, 3, padding='valid')(y)
    y = layers.GroupNormalization(groups=filters)(y)
    y = layers.ReLU()(y)
    y = ReflectionPadding2D()(y)
    y = layers.Conv2D(filters, 3, padding='valid')(y)
    y = layers.GroupNormalization(groups=filters)(y)
    return layers.Add()([x, y])

def build_generator():
    ip = Input(shape=(400, 400, 1))
    x = ReflectionPadding2D((3, 3))(ip)
    x = layers.Conv2D(64, 7, padding='valid')(x)
    x = layers.GroupNormalization(groups=64)(x)
    feat1 = layers.ReLU()(x)

    x = ReflectionPadding2D((1, 1))(feat1)
    x = layers.Conv2D(128, 3, strides=2, padding='valid')(x)
    x = layers.GroupNormalization(groups=128)(x)
    feat2 = layers.ReLU()(x)

    x = ReflectionPadding2D((1, 1))(feat2)
    x = layers.Conv2D(256, 3, strides=2, padding='valid')(x)
    x = layers.GroupNormalization(groups=256)(x)
    feat3 = layers.ReLU()(x)
    
    x = ReflectionPadding2D((1, 1))(feat3)
    x = layers.Conv2D(256, 3, strides=1, padding='valid')(x) 
    x = layers.GroupNormalization(groups=256)(x)
    x = layers.ReLU()(x)

    for _ in range(9): x = residual_block(x, 256)

    x = layers.Conv2DTranspose(256, 3, strides=1, padding='same')(x)
    x = layers.GroupNormalization(groups=256)(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding='same')(x)
    x = layers.GroupNormalization(groups=128)(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2DTranspose(64, 3, strides=2, padding='same')(x)
    x = layers.GroupNormalization(groups=64)(x)
    x = layers.ReLU()(x)

    x = ReflectionPadding2D((3, 3))(x)
    x = layers.Conv2D(64, 7, padding='valid')(x)
    x = layers.GroupNormalization(groups=64)(x)
    last_feat = layers.ReLU()(x)
    
    out = layers.Conv2D(1, 7, padding='same', activation='sigmoid', dtype='float32')(last_feat)
    return Model(ip, [out, feat1, feat2, feat3, last_feat])

print("3. Loading Trained Weights...")
# 3. LOAD THE TRAINED WEIGHTS
generator = build_generator()

# ---> REPLACE THIS PATH WITH YOUR ACTUAL SAVED KAGGLE OUTPUT PATH <---
weights_path = "/kaggle/input/notebooks/yashrnitp/domain-adaptation-using-gan/harmonization_gen_weights_FINAL.weights.h5" 
generator.load_weights(weights_path)

print("Setup Complete! The generator and dataset are ready.")

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(15, 45))

# Pull 10 pairs of images from the dataset
for i, (sample_a, sample_b) in enumerate(train_dataset.take(10)):
    
    # Generate the adapted image
    generated = generator(sample_a, training=False)[0]
    
    # 1. Plot Source (Domain A)
    plt.subplot(10, 3, i*3 + 1)
    if i == 0: plt.title("Source (Domain A)", fontsize=20, fontweight='bold')
    plt.imshow(sample_a[0, :, :, 0], cmap='gray')
    plt.axis('off')
    
    # 2. Plot Generated Image
    plt.subplot(10, 3, i*3 + 2)
    if i == 0: plt.title("Generated (Adapted)", fontsize=20, fontweight='bold')
    plt.imshow(generated[0, :, :, 0], cmap='gray')
    plt.axis('off')
    
    # 3. Plot Target Style (Domain B)
    plt.subplot(10, 3, i*3 + 3)
    if i == 0: plt.title("Target Style (Domain B)", fontsize=20, fontweight='bold')
    plt.imshow(sample_b[0, :, :, 0], cmap='gray')
    plt.axis('off')


plt.tight_layout()
plt.savefig("/kaggle/working/adaptation_10_results.png", bbox_inches='tight', dpi=150)
plt.show()

print("10 images successfully generated and saved as 'adaptation_10_results.png'!")

In [ ]:
import numpy as np
import tensorflow as tf

print("Calculating Quantitative Metrics (SSIM, HC, and BD)...")
print("Evaluating 100 images. This will take about 15-30 seconds...\n")

ssim_scores = []
hc_baseline_scores = []
hc_adapted_scores = []
bd_baseline_scores = []
bd_adapted_scores = []

# Evaluate on 100 random image pairs
for i, (sample_a, sample_b) in enumerate(train_dataset.take(100)):
    
    # 1. Generate the adapted image 
    generated = generator(sample_a, training=False)[0] 
    
    # 2. CALCULATE SSIM (Anatomy Preservation)
    ssim_val = tf.image.ssim(sample_a, generated, max_val=1.0)
    ssim_scores.append(ssim_val.numpy()[0])
    
    # Extract numpy arrays for histogram math
    img_a = sample_a[0, :, :, 0].numpy().flatten()
    img_b = sample_b[0, :, :, 0].numpy().flatten()
    img_gen = generated[0, :, :, 0].numpy().flatten()
    
    # 3. HISTOGRAMS
    hist_a, _ = np.histogram(img_a, bins=256, range=(0, 1))
    hist_b, _ = np.histogram(img_b, bins=256, range=(0, 1))
    hist_gen, _ = np.histogram(img_gen, bins=256, range=(0, 1))
    
    # Normalize histograms to sum to 1.0
    hist_a = hist_a / np.sum(hist_a)
    hist_b = hist_b / np.sum(hist_b)
    hist_gen = hist_gen / np.sum(hist_gen)
    
    # CALCULATE HISTOGRAM CORRELATION (HC) -> Higher is better
    baseline_corr = np.corrcoef(hist_a, hist_b)[0, 1]
    hc_baseline_scores.append(baseline_corr)
    
    adapted_corr = np.corrcoef(hist_gen, hist_b)[0, 1]
    hc_adapted_scores.append(adapted_corr)
    
    # CALCULATE BHATTACHARYYA DISTANCE (BD) -> Lower is better
    # Math: sqrt( 1 - sum( sqrt(hist1 * hist2) ) )
    # Note: np.maximum(0, ...) prevents floating point errors from causing math domain crashes
    baseline_bd = np.sqrt(np.maximum(0.0, 1.0 - np.sum(np.sqrt(hist_a * hist_b))))
    bd_baseline_scores.append(baseline_bd)
    
    adapted_bd = np.sqrt(np.maximum(0.0, 1.0 - np.sum(np.sqrt(hist_gen * hist_b))))
    bd_adapted_scores.append(adapted_bd)

# Calculate Means and Standard Deviations (SD)
mean_ssim, sd_ssim = np.mean(ssim_scores), np.std(ssim_scores)

mean_hc_base, sd_hc_base = np.mean(hc_baseline_scores), np.std(hc_baseline_scores)
mean_hc_adapt, sd_hc_adapt = np.mean(hc_adapted_scores), np.std(hc_adapted_scores)

mean_bd_base, sd_bd_base = np.mean(bd_baseline_scores), np.std(bd_baseline_scores)
mean_bd_adapt, sd_bd_adapt = np.mean(bd_adapted_scores), np.std(bd_adapted_scores)

# Print the Final Scientific Table!
print("="*70)
print("         FINAL QUANTITATIVE RESULTS (COMPARED TO PAPER)         ")
print("="*70)
print(f"1. Anatomy Preservation (Whole Image SSIM):")
print(f"   -> Your Model: {mean_ssim:.2f} ({sd_ssim:.2f})")
print(f"   -> Paper's Model (Table 3): ~0.53 (0.09)")
print("-" * 70)
print(f"2. Texture Correlation (HC) [Higher is closer to 1.0]:")
print(f"   -> BEFORE AI (Domain A vs B): {mean_hc_base:.3f} ({sd_hc_base:.3f})")
print(f"   -> AFTER AI (Generated vs B): {mean_hc_adapt:.3f} ({sd_hc_adapt:.3f})")
print("-" * 70)
print(f"3. Bhattacharyya Distance (BD) [Lower is closer to 0.0]:")
print(f"   -> BEFORE AI (Domain A vs B): {mean_bd_base:.3f} ({sd_bd_base:.3f})")
print(f"   -> AFTER AI (Generated vs B): {mean_bd_adapt:.3f} ({sd_bd_adapt:.3f})")
print("="*70)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric

print("Generating SSIM Map...")

# 1. Grab 1 sample image from your dataset
sample_a, _ = next(iter(train_dataset.take(1)))

# 2. Generate the adapted image
generated = generator(sample_a, training=False)[0]

# 3. Convert tensors to 2D numpy arrays (remove batch and channel dimensions)
img_in = sample_a[0, :, :, 0].numpy()
img_out = generated[0, :, :, 0].numpy()

# 4. Calculate local SSIM map
# 'full=True' is the secret here; it returns the 2D pixel-by-pixel similarity matrix
score, ssim_map = ssim_metric(img_in, img_out, data_range=1.0, full=True)

# 5. Plotting (Recreating the 3-panel layout from the paper)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Input Image
axes[0].imshow(img_in, cmap='gray')
axes[0].set_title("Input image (US system A)", fontsize=16, pad=15)
axes[0].axis('off')

# (b) Generated Image
axes[1].imshow(img_out, cmap='gray')
axes[1].set_title("Generated image (US system B)", fontsize=16, pad=15)
axes[1].axis('off')

# (c) SSIM Map
# The 'RdYlGn' colormap matches the paper: Red (-1) -> Yellow (0) -> Green (+1)
im = axes[2].imshow(ssim_map, cmap='RdYlGn', vmin=-1.0, vmax=1.0)
axes[2].set_title("SSIM map", fontsize=16, pad=15)
axes[2].axis('off')

# Add the exact colorbar shown in the paper
cbar = fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
cbar.set_label('Similarity', rotation=90)

plt.tight_layout()
plt.savefig("/kaggle/working/paper_figure_7_replication.png", bbox_inches='tight', dpi=150)
plt.show()

print(f"Overall Image SSIM Score: {score:.3f}")

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("Calculating Medical Noise Reduction (Contrast dB)...")

# 1. FIND THE DIRECTORIES BASED ON YOUR SCREENSHOTS
us_dir = None
mask_dir = None

# Scan Kaggle directories for the specific folders
for root, dirs, files in os.walk("/kaggle/input"):
    if os.path.basename(root) == "US images":
        us_dir = root
    elif os.path.basename(root) == "Expert mask images":
        mask_dir = root

if not us_dir or not mask_dir:
    raise ValueError("Could not find the 'US images' or 'Expert mask images' folders!")

# 2. MATCH THE EXACT FILENAMES
# Grab the first image in the US images folder
valid_files = [f for f in os.listdir(us_dir) if f.endswith(('.png', '.jpg'))]
filename = valid_files[0] # You can change 0 to another number to test different images!

img_path = os.path.join(us_dir, filename)
mask_path = os.path.join(mask_dir, filename)

print(f"Successfully Paired:")
print(f"Image: {img_path}")
print(f"Mask:  {mask_path}")

# 3. LOAD AND PREPROCESS
def load_img(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=1, expand_animations=False)
    img = tf.image.resize(img, [400, 400])
    return tf.cast(img, tf.float32) / 255.0

original_img_tensor = load_img(img_path)
mask_tensor = load_img(mask_path)

# Pass the original image through your trained AI generator
input_batch = tf.expand_dims(original_img_tensor, 0)
generated_img_tensor = generator(input_batch, training=False)[0]

# Convert to 2D numpy arrays
original_img = original_img_tensor[:, :, 0].numpy()
generated_img = generated_img_tensor[0, :, :, 0].numpy()
mask = mask_tensor[:, :, 0].numpy()

# ---------------------------------------------------------
# 4. DEFINE THE REGIONS OF INTEREST (ROIs)
# In this dataset, the Expert Mask highlights the Lumen (blood) in WHITE.
# Therefore, Lumen = Mask pixels > 0.5
# We will define the Adventitia (Wall) as the tissue around the lumen 
# (Mask < 0.5, but we exclude pitch-black background pixels < 0.05 so they don't break the math)
# ---------------------------------------------------------

lumen_roi = mask > 0.5
adventitia_roi = (mask < 0.5) & (original_img > 0.05) 

# 5. EQUATION 10: CONTRAST (dB) MATH
def calculate_contrast_db(image, lumen_bool, wall_bool):
    lumen_pixels = image[lumen_bool]
    wall_pixels = image[wall_bool]
    
    mean_lumen = np.mean(lumen_pixels)
    mean_wall = np.mean(wall_pixels)
    
    # Safety check to prevent log(0)
    if mean_lumen <= 0 or mean_wall <= 0:
        return np.nan
        
    # Equation 10: 20 * log10( Lumen / Adventitia )
    return 20.0 * np.log10(mean_lumen / mean_wall)

# Calculate the scores
baseline_contrast = calculate_contrast_db(original_img, lumen_roi, adventitia_roi)
adapted_contrast = calculate_contrast_db(generated_img, lumen_roi, adventitia_roi)

# 6. PRINT THE RESULTS
print("\n" + "="*60)
print("    MEDICAL RISK MARKER: NOISE REDUCTION (CONTRAST dB)    ")
print("="*60)
print(f"BEFORE AI (Original Image) : {baseline_contrast:.2f} dB")
print(f"AFTER AI (Generated Image) : {adapted_contrast:.2f} dB")
print("-" * 60)
print("NOTE: A more NEGATIVE number means the Lumen is darker")
print("compared to the wall (i.e., NOISE WAS REDUCED).")
print("="*60)

# 7. VISUALIZE THE TISSUES
plt.figure(figsize=(15, 5))

plt.subplot(1, 4, 1)
plt.title("Original Image")
plt.imshow(original_img, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title("Generated Image")
plt.imshow(generated_img, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title("Lumen ROI (Blood)")
plt.imshow(original_img, cmap='gray')
plt.imshow(lumen_roi, cmap='Reds', alpha=0.4) # Overlay in red
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title("Wall ROI (Tissue)")
plt.imshow(original_img, cmap='gray')
plt.imshow(adventitia_roi, cmap='Blues', alpha=0.4)
plt.axis('off')

plt.tight_layout()
plt.savefig("/kaggle/working/medical_roi_masks.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
import os
import numpy as np
import tensorflow as tf

print("Calculating Clinical Risk Marker (GSM) and Re-classification Rate...")
print("Processing 50 images. This may take a minute...\n")

# 1. FIND THE DIRECTORIES
us_dir = None
mask_dir = None
for root, dirs, files in os.walk("/kaggle/input"):
    if os.path.basename(root) == "US images":
        us_dir = root
    elif os.path.basename(root) == "Expert mask images":
        mask_dir = root

valid_files = [f for f in os.listdir(us_dir) if f.endswith(('.png', '.jpg'))]

def load_img(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=1, expand_animations=False)
    img = tf.image.resize(img, [400, 400])
    return tf.cast(img, tf.float32) / 255.0

gsm_before_list = []
gsm_after_list = []
reclassified_count = 0
valid_images_processed = 0

# 2. PROCESS 50 IMAGES
for filename in valid_files[:50]:
    img_path = os.path.join(us_dir, filename)
    mask_path = os.path.join(mask_dir, filename)
    
    if not os.path.exists(mask_path):
        continue

    # Load images
    original_img_tensor = load_img(img_path)
    mask_tensor = load_img(mask_path)

    # Generate AI image
    input_batch = tf.expand_dims(original_img_tensor, 0)
    generated_img_tensor = generator(input_batch, training=False)[0]

    # Convert to numpy and scale to 0-255 (GSM requires 8-bit values!)
    original_img = original_img_tensor[:, :, 0].numpy() * 255.0
    generated_img = generated_img_tensor[0, :, :, 0].numpy() * 255.0
    mask = mask_tensor[:, :, 0].numpy()

    # Define ROIs (Lumen = white mask, Tissue = dark mask)
    lumen_roi = mask > 0.5
    tissue_roi = (mask < 0.5) & (original_img > 10.0) # >10 removes dead black background
    
    # Skip if mask is broken or empty
    if np.sum(lumen_roi) < 10 or np.sum(tissue_roi) < 10:
        continue

    # 3. GSM CALCULATOR FUNCTION (Equation 9/11)
    def calculate_gsm(image, lumen_bool, tissue_bool):
        lumen_pixels = image[lumen_bool]
        tissue_pixels = image[tissue_bool]
        
        mean_lumen = np.mean(lumen_pixels)
        max_tissue = np.max(tissue_pixels)
        
        # Prevent division by zero
        if max_tissue == 0:
            return np.nan
            
        # Apply the paper's standardization formula to the tissue pixels
        mapped_pixels = 190.0 * (tissue_pixels - mean_lumen) / max_tissue
        
        # GSM is the median of these mapped pixels
        return np.median(mapped_pixels)

    # Calculate GSM Before and After
    gsm_before = calculate_gsm(original_img, lumen_roi, tissue_roi)
    gsm_after = calculate_gsm(generated_img, lumen_roi, tissue_roi)
    
    if np.isnan(gsm_before) or np.isnan(gsm_after):
        continue

    gsm_before_list.append(gsm_before)
    gsm_after_list.append(gsm_after)
    
    # 4. RE-CLASSIFICATION CHECK (Threshold = 25)
    class_before = "Stable" if gsm_before >= 25 else "Vulnerable"
    class_after = "Stable" if gsm_after >= 25 else "Vulnerable"
    
    if class_before != class_after:
        reclassified_count += 1
        
    valid_images_processed += 1

# Calculate Final Statistics
mean_gsm_before, sd_gsm_before = np.mean(gsm_before_list), np.std(gsm_before_list)
mean_gsm_after, sd_gsm_after = np.mean(gsm_after_list), np.std(gsm_after_list)
reclass_rate = (reclassified_count / valid_images_processed) * 100

# Print Table 4 / 7 format
print("="*65)
print("     CLINICAL IMPACT: GSM & RE-CLASSIFICATION RATE (TABLE 4)    ")
print("="*65)
print(f"Total Images Evaluated: {valid_images_processed}")
print("-" * 65)
print("Grey Scale Median (GSM) Score:")
print(f"   -> BEFORE AI: {mean_gsm_before:.1f} ({sd_gsm_before:.1f})")
print(f"   -> AFTER AI:  {mean_gsm_after:.1f} ({sd_gsm_after:.1f})")
print("-" * 65)
print(f"Patients Re-Classified (Crossed GSM=25 threshold):")
print(f"   -> {reclassified_count} out of {valid_images_processed} patients ({reclass_rate:.1f}%)")
print("="*65)
if reclass_rate > 0:
    print("WARNING: The AI changed the clinical diagnosis for some patients!")
    print("This perfectly matches the paper's findings.")
else:
    print("The AI preserved the clinical diagnosis perfectly in this batch.")
print("="*65)